# EoMT fine-tuning on Cityscapes — Step 5

Fine-tuning of an EoMT model pre-trained on COCO panoptic, adapted to Cityscapes semantic segmentation (19 classes). This notebook is compatible with the first 3 configurations of the fine-tuning process explained in the report. For the last config refer to the step5_config_blocks_10_11 notebook, since it deals with discriminative learning rates and code changes a bit.


## Setup


In [ ]:
# Change dir to  project directory path
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
%cd /content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject/eomt


In [ ]:
!pip install -q -r requirements.txt


In [ ]:
import wandb
from google.colab import userdata
wandb.login(key=userdata.get('WANDB_API_KEY'))


## Imports and configuration


In [ ]:
import os
import importlib
import warnings
import time

import yaml
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp.autocast_mode import autocast
from torch.amp.grad_scaler import GradScaler
from tqdm import tqdm
from lightning import seed_everything

seed_everything(0, verbose=False)
device = 0

# Cityscapes semantic config (19 classes, 640px input).
config_path = "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

data_path = "/content/drive/MyDrive/Fundamentals_Progetto/Coding_Part_Project"


## Dataset (Cityscapes)


In [ ]:
# Build the Cityscapes data module from the config.
data_module_name, class_name = config["data"]["class_path"].rsplit(".", 1)
data_module = getattr(importlib.import_module(data_module_name), class_name)
data_module_kwargs = config["data"].get("init_args", {})

data = data_module(
    path=data_path,
    batch_size=2,
    num_workers=2,
    check_empty_targets=False,
    **data_module_kwargs,
).setup()

print(f"Train samples: {len(data.train_dataloader().dataset)}")
print(f"Val samples:   {len(data.val_dataloader().dataset)}")
print(f"Num classes:   {data.num_classes}")
print(f"Image size:    {data.img_size}")


## Model


In [ ]:
# Build the EoMT model with the Cityscapes head (19 classes).
warnings.filterwarnings(
    "ignore",
    message=r".*Attribute 'network' is an instance of `nn\.Module`.*",
)

encoder_cfg = config["model"]["init_args"]["network"]["init_args"]["encoder"]
encoder_module_name, encoder_class_name = encoder_cfg["class_path"].rsplit(".", 1)
encoder_cls = getattr(importlib.import_module(encoder_module_name), encoder_class_name)
encoder = encoder_cls(img_size=data.img_size, **encoder_cfg.get("init_args", {}))

network_cfg = config["model"]["init_args"]["network"]
network_module_name, network_class_name = network_cfg["class_path"].rsplit(".", 1)
network_cls = getattr(importlib.import_module(network_module_name), network_class_name)
network_kwargs = {k: v for k, v in network_cfg["init_args"].items() if k != "encoder"}
network = network_cls(
    masked_attn_enabled=False,
    num_classes=data.num_classes,
    encoder=encoder,
    **network_kwargs,
)

lit_module_name, lit_class_name = config["model"]["class_path"].rsplit(".", 1)
lit_cls = getattr(importlib.import_module(lit_module_name), lit_class_name)
model_kwargs = {k: v for k, v in config["model"]["init_args"].items() if k != "network"}
if "stuff_classes" in config["data"].get("init_args", {}):
    model_kwargs["stuff_classes"] = config["data"]["init_args"]["stuff_classes"]
model = lit_cls(network=network, **model_kwargs).to(f"cuda:{device}")
print("Model built.")


Load the COCO-pretrained weights into the Cityscapes model. Shape-incompatible layers (i.e. the 133-class classification head) are dropped and reinitialized; everything else transfers.


In [ ]:
# Load the COCO-pretrained weights, filtered by shape.
# The 133-class COCO classification head does not match the 19-class Cityscapes
# head and is therefore discarded; the rest of the network transfers.


# CHANGE TO YOUR COCO WEIGHTS PATH
coco_weights_path = (
    "/content/drive/MyDrive/Fundamentals_Progetto/Coding_Part_Project/"
    "CourseProjectAnomaly/eomt_coco.bin"
)

state_dict = torch.load(coco_weights_path, map_location=f"cuda:{device}", weights_only=True)
model_state_dict = model.state_dict()

filtered_state_dict = {}
shape_mismatch = []
unexpected_keys = []
for name, param in state_dict.items():
    if name not in model_state_dict:
        unexpected_keys.append(name)
        continue
    if param.shape == model_state_dict[name].shape:
        filtered_state_dict[name] = param
    else:
        shape_mismatch.append(name)

load_info = model.load_state_dict(filtered_state_dict, strict=False)
print(f"Transferred from COCO: {len(filtered_state_dict)} tensors")
print(f"Shape mismatch (discarded, e.g. class_head): {len(shape_mismatch)}")
print(f"Randomly initialized (missing in COCO):     {len(load_info.missing_keys)}")


## Inference and evaluation utilities


In [ ]:
IGNORE_INDEX = 255


def infer_semantic(img, target):
    """Sliding-window semantic inference on a single image."""
    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        imgs = [img.to(f"cuda:{device}")]
        img_sizes = [img.shape[-2:] for img in imgs]
        crops, origins = model.window_imgs_semantic(imgs)

        mask_logits_per_layer, class_logits_per_layer = model(crops)
        mask_logits = F.interpolate(
            mask_logits_per_layer[-1], model.img_size, mode="bilinear"
        )
        crop_logits = model.to_per_pixel_logits_semantic(
            mask_logits, class_logits_per_layer[-1]
        )
        logits = model.revert_window_logits_semantic(crop_logits, origins, img_sizes)
        preds = logits[0].argmax(0).cpu()

    pred_array = preds.numpy()
    target_array = model.to_per_pixel_targets_semantic([target], IGNORE_INDEX)[0].numpy()
    return pred_array, target_array


In [ ]:
def evaluate_and_print(model, dataloader):
    """Compute mIoU on Cityscapes val (19 classes, ignore_index=255 masked out)."""
    intersections = np.zeros(19)
    unions = np.zeros(19)

    for batch in tqdm(dataloader, desc="Eval"):
        img, target = batch
        if isinstance(img, list):
            img = img[0]
        if isinstance(target, list):
            target = target[0]

        pred_array, target_array = infer_semantic(img, target)
        valid_mask = (target_array != IGNORE_INDEX)

        for cls in range(19):
            pred_mask = (pred_array == cls)
            target_mask = (target_array == cls)
            intersections[cls] += (pred_mask & target_mask & valid_mask).sum()
            unions[cls]        += ((pred_mask | target_mask) & valid_mask).sum()

    ious = []
    for cls in range(19):
        if unions[cls] > 0:
            ious.append(intersections[cls] / unions[cls])
    miou = float(np.mean(ious)) if ious else 0.0
    print(f"mIoU: {miou:.4f}")
    return miou


## Freezing configuration
This is the cell where we decide what layers to unfreeze (in this case head + upscale)




In [ ]:
# Freezing configuration: head + upscale (no queries, no DINOv2 blocks)
# Trainable: class_head, mask_head, upscale (+ pos_embed)

# 1) Freeze everything.
for param in model.parameters():
    param.requires_grad = False

# 2) Unfreeze the prediction heads and the mask upsampling.
TRAINABLE_PREFIXES = [
    "network.class_head",
    "network.mask_head",
    "network.upscale",
]
trainable_param_names = []
for name, param in model.named_parameters():
    if any(name.startswith(p) for p in TRAINABLE_PREFIXES):
        param.requires_grad = True
        trainable_param_names.append(name)

# 3) pos_embed is interpolated to the Cityscapes resolution (not in its
#    pretrained-optimal state), so we unfreeze it in all configurations
#    for consistency across the ablation.
for name, param in model.named_parameters():
    if name == "network.encoder.backbone.pos_embed":
        param.requires_grad = True
        trainable_param_names.append(name)

total_params = sum(p.numel() for p in model.parameters())
trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable: {trainable_count:,} / {total_params:,} ({100*trainable_count/total_params:.2f}%)")
for n in trainable_param_names:
    print(f"  - {n}")


## Optimizer, scheduler, and W&B


In [ ]:
# Training configuration.
# CONFIG_NAME is the single source of truth: W&B run id/name and the
# checkpoint subdirectory both derive from it, so concurrent runs of
# different configurations do not collide.
CONFIG_NAME = "config-head-upscale"

LEARNING_RATE = 1e-4    # Initial learning rate
WEIGHT_DECAY  = 0.05
ETA_MIN       = 1e-6    # Final learning rate

TOTAL_EPOCHS    = 10   # Full plan (constant across resume sessions).
EPOCHS_THIS_RUN = 4    # How many epochs to run in this Colab session.

BATCH_SIZE          = 2
EVAL_EVERY_N_EPOCHS = 1

# Dir containing the weights of the checkpoints
checkpoint_base = "/content/drive/MyDrive/Fundamentals_Progetto/Coding_Part_Project/checkpoints"
checkpoint_dir  = os.path.join(checkpoint_base, CONFIG_NAME)
os.makedirs(checkpoint_dir, exist_ok=True)
RESUME_PATH = os.path.join(checkpoint_dir, "training_state_latest.pt")

optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.999),
)
# Cosine annealing over the full 10-epoch plan. Scheduler state is saved
# and restored across sessions, so the LR is continuous across resumes.
scheduler = CosineAnnealingLR(optimizer, T_max=TOTAL_EPOCHS, eta_min=ETA_MIN)
scaler = GradScaler()

wandb.init(
    project="step-5-finetuning",
    name=CONFIG_NAME,
    id=CONFIG_NAME,
    resume="allow",
    config={
        "config_name":   CONFIG_NAME,
        "learning_rate": LEARNING_RATE,
        "weight_decay":  WEIGHT_DECAY,
        "eta_min":       ETA_MIN,
        "total_epochs":  TOTAL_EPOCHS,
        "batch_size":    BATCH_SIZE,
        "dataset":       "Cityscapes",
        "num_classes":   19,
        "optimizer":     "AdamW",
        "scheduler":     "CosineAnnealingLR",
    },
)
wandb.define_metric("epoch")
wandb.define_metric("epoch/*", step_metric="epoch")
wandb.define_metric("val/*",   step_metric="epoch")

print(f"Setup ready: {CONFIG_NAME}")
print(f"Cosine: {LEARNING_RATE} -> {ETA_MIN} over {TOTAL_EPOCHS} epochs")
print(f"Checkpoint dir: {checkpoint_dir}")


## Resume from checkpoint (if any)


In [ ]:
def save_full_state(path, epochs_done, miou=None):
    """Save the full training state (model + optimizer + scheduler + epoch)."""
    torch.save({
        "model":       model.state_dict(),
        "optimizer":   optimizer.state_dict(),
        "scheduler":   scheduler.state_dict(),
        "epochs_done": epochs_done,
        "miou":        miou,
    }, path)


if os.path.exists(RESUME_PATH):
    print(f"Resuming from checkpoint: {RESUME_PATH}")
    ckpt = torch.load(RESUME_PATH, map_location=f"cuda:{device}", weights_only=False)
    model.load_state_dict(ckpt["model"], strict=False)
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    epochs_done = ckpt["epochs_done"]
    print(f"  Epochs already completed: {epochs_done}/{TOTAL_EPOCHS}")
    for gi, g in enumerate(optimizer.param_groups):
        print(f"  Current LR (group {gi}): {g['lr']:.2e}")
else:
    print("No checkpoint found: starting fresh from the loaded COCO weights.")
    epochs_done = 0


## Training


In [ ]:
# Training dataloader.
train_loader = data.train_dataloader()
print(f"Train batches per epoch: {len(train_loader)} (batch_size={BATCH_SIZE})")


In [ ]:
def move_batch_to_device(batch, device):
    """Move a (possibly nested) batch dict/list/tensor to the target device."""
    if torch.is_tensor(batch):
        return batch.to(device, non_blocking=True)
    if isinstance(batch, dict):
        return {k: move_batch_to_device(v, device) for k, v in batch.items()}
    if isinstance(batch, (list, tuple)):
        return type(batch)(move_batch_to_device(v, device) for v in batch)
    return batch


def train_one_epoch(model, train_loader, optimizer, scaler, epoch, num_epochs):
    """One training epoch: AMP + gradient clipping."""
    model.train()
    running_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    for step, batch in enumerate(pbar):
        batch = move_batch_to_device(batch, f"cuda:{device}")
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
            loss = model.training_step(batch, step)

        if torch.isnan(loss) or torch.isinf(loss):
            print(f"  Warning: skipping step {step} (loss={loss.item()})")
            continue

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(
            (p for p in model.parameters() if p.requires_grad),
            max_norm=1.0,
        )
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        avg_loss = running_loss / (step + 1)
        current_lr = optimizer.param_groups[0]["lr"]
        pbar.set_postfix(loss=f"{loss.item():.4f}", avg_loss=f"{avg_loss:.4f}", lr=f"{current_lr:.2e}")
        wandb.log({"batch/loss": loss.item(), "batch/lr": current_lr})

    return running_loss / max(len(train_loader), 1)


In [ ]:
loss_history = []
miou_history = []
start_time = time.time()

end_epoch = min(epochs_done + EPOCHS_THIS_RUN, TOTAL_EPOCHS)
if epochs_done >= TOTAL_EPOCHS:
    print(f"Training already complete ({epochs_done}/{TOTAL_EPOCHS}). Nothing to do.")
else:
    print(f"This session: epochs {epochs_done+1} -> {end_epoch} (of {TOTAL_EPOCHS} total)\n")

for epoch in range(epochs_done, end_epoch):
    absolute_epoch = epoch + 1

    epoch_start = time.time()
    avg_loss = train_one_epoch(
        model=model, train_loader=train_loader, optimizer=optimizer,
        scaler=scaler, epoch=epoch, num_epochs=TOTAL_EPOCHS,
    )
    epoch_time = time.time() - epoch_start
    loss_history.append(avg_loss)

    # Scheduler step: once per epoch, after training.
    scheduler.step()
    current_lr = optimizer.param_groups[0]["lr"]

    elapsed = time.time() - start_time
    done_this_run = epoch - epochs_done + 1
    eta = elapsed / done_this_run * (end_epoch - epochs_done) - elapsed
    print(f"\n=== Epoch {absolute_epoch}/{TOTAL_EPOCHS} done ===")
    print(f"  Avg loss:     {avg_loss:.4f}")
    print(f"  Next LR:      {current_lr:.2e}")
    print(f"  Epoch time:   {epoch_time/60:.1f} min")
    print(f"  Session ETA:  {eta/60:.1f} min")

    log_dict = {"epoch/avg_loss": avg_loss, "epoch/lr": current_lr, "epoch": absolute_epoch}

    # Validation mIoU.
    val_miou = None
    if absolute_epoch % EVAL_EVERY_N_EPOCHS == 0 or absolute_epoch == TOTAL_EPOCHS:
        print(f"\n--- Validation (epoch {absolute_epoch}) ---")
        model.eval()
        val_miou = evaluate_and_print(model, data.val_dataloader())
        model.train()
        miou_history.append((absolute_epoch, val_miou))
        log_dict["val/mIoU"] = val_miou

    wandb.log(log_dict)

    # Save full training state.
    save_full_state(RESUME_PATH, absolute_epoch, val_miou)
    epoch_ckpt = os.path.join(checkpoint_dir, f"training_state_epoch{absolute_epoch}.pt")
    save_full_state(epoch_ckpt, absolute_epoch, val_miou)
    print(f"  Checkpoint saved (epoch {absolute_epoch})")

print(f"\nSession completed in {(time.time()-start_time)/60:.1f} min")
if loss_history:
    print(f"  Loss: {loss_history[0]:.4f} -> {loss_history[-1]:.4f}")


## Final evaluation


In [ ]:
# Final evaluation (unless it already happened in the last epoch of the loop).
if miou_history and miou_history[-1][0] == TOTAL_EPOCHS:
    print(f"Final mIoU (epoch {TOTAL_EPOCHS}): {miou_history[-1][1]:.4f}")
else:
    model.eval()
    print("\n--- Final mIoU on validation set ---")
    val_miou = evaluate_and_print(model, data.val_dataloader())
    print(f"mIoU: {val_miou:.4f}")

# Close the W&B session. The run remains visible on the dashboard;
# resume="allow" will reopen it in the next session if training continues.
wandb.finish()
print("W&B session closed.")
